# Cosine Similarity — AI Clothing Recommendation

### Project Goal
Cosine Similarity is used to compare users based on their clothing preferences and find users with similar tastes.

**Pipeline:** User Preferences → One-Hot Encoding → User Vectors → Cosine Similarity → Most Similar Users

> **Note:** The current 100-user dataset is synthetic data used to test the algorithm. It should be replaced by real user responses when the website is connected to the system.

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity


## 1. Load the Dataset

In [ ]:
df = pd.read_csv("user_preferences_100.csv")

print("Number of users:", len(df))
df.head()

## 2. Select User Preference Features

We use the eight questions that describe the user's clothing taste. User ID is not used as a feature because it identifies the user rather than describing their preferences.

In [ ]:
features = [
    "preferred_style",
    "preferred_clothing_type",
    "favorite_color",
    "color_combination",
    "preferred_fit",
    "usual_outfit",
    "fashion_trends_importance",
    "preferred_occasion"
]

X = df[features]
X.head()

## 3. Convert Answers into Numerical Vectors

The answers are categorical, so One-Hot Encoding converts them into numerical features that can be compared mathematically.

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_encoded = encoder.fit_transform(X)

print("Original shape:", X.shape)
print("Encoded shape:", X_encoded.shape)

## 4. Calculate Cosine Similarity

Cosine Similarity measures the angle between two user vectors. A value closer to **1** means more similar preferences, while a value closer to **0** means less similarity.

In [ ]:
similarity_matrix = cosine_similarity(X_encoded)

print("Similarity matrix shape:", similarity_matrix.shape)

## 5. Find the Most Similar Users

In [ ]:
def find_similar_users(user_id, top_n=5):
    user_index = df.index[df["user_id"] == user_id][0]
    scores = similarity_matrix[user_index]

    # Sort from highest to lowest similarity
    similar_indices = scores.argsort()[::-1]

    # Remove the selected user from their own results
    similar_indices = [i for i in similar_indices if i != user_index]
    top_indices = similar_indices[:top_n]

    result = df.iloc[top_indices].copy()
    result["similarity_score"] = scores[top_indices]

    return result[["user_id"] + features + ["similarity_score"]]

## 6. Test the Algorithm

In [ ]:
user_id = 1

print(f"Most similar users to User {user_id}:")
find_similar_users(user_id, top_n=5)

## 7. Interpretation

The returned users have the highest Cosine Similarity scores compared with the selected user. These similar users can later be used as part of the recommendation system to identify clothing preferences that may be relevant to the current user.

### Example
If User 1 has a similarity score of 0.625 with User 72, their preference vectors have a cosine similarity of 0.625 based on the eight preference categories used in this experiment.

### Future Integration
When the website is connected, the same process can be applied to a new user's answers instead of using the synthetic users only.